# The follwoign codes focus on the extraction of the planned

1) Merge multiple final dataframes into one combined file.

2) Add derived features (HR, BP, RR, SBP, DBP, etc.) and apply quality checks.

3) Remove low-quality PPG cases (based on SQI).

4) Exclude rows with mislabeled signals.

5) Stratify subjects by age, diagnosis codes, and other factors.

6) Build stratified folds for downstream training and evaluation.

## Merging the dataframe files to a create a merged file

In [ ]:
import os
import pandas as pd
import re

# Set your folder path
folder_path = 'path/to/folders/of/pkl/files'

# List and filter only .pkl files
files = os.listdir(folder_path)
pkl_files = [f for f in files if f.endswith('.pkl')]

# Sort files based on the numbers in the filename
def extract_numbers(f):
    match = re.search(r'df_(\d+)_(\d+)_\d+\_temp.pkl', f)
    return (int(match.group(1)), int(match.group(2))) if match else (float('inf'), float('inf'))



pkl_files_sorted = sorted(pkl_files, key=extract_numbers)

# Print the order of files to be merged
print("Order of files being merged:")
for f in pkl_files_sorted:
    print(f)

# Load and merge DataFrames
df_list = [pd.read_pickle(os.path.join(folder_path, f)) for f in pkl_files_sorted]
merged_df = pd.concat(df_list, ignore_index=True)

# Optional: Save the merged DataFrame
merged_df.to_pickle('path/to merged pkl file.pkl')

print("\nMerged DataFrame shape:", merged_df.shape)

## Now addig extra coloumns e.g., hr, bp, rr, sbp, dbp,..

In [ ]:
import pandas as pd
import numpy as np

df_events=df1


# Helper function to check if all elements in a list are not nan
def all_not_nan(x):
    return isinstance(x, (list, np.ndarray)) and all(not pd.isna(i) for i in x)

# Create new columns
df_events['bp'] = df_events.apply(lambda row: int(all_not_nan(row['vector_10s_sbp']) and all_not_nan(row['vector_10s_dbp'])), axis=1)
df_events['hr'] = df_events.apply(lambda row: int(all_not_nan(row['vector_10s_hr'])), axis=1)
df_events['rr'] = df_events['median_30s_rr'].apply(lambda x: int(not pd.isna(x)))



df=df_events

def strict_median(x):
    if isinstance(x, (list, np.ndarray)) and not any(pd.isna(x)):
        return np.median(x)
    return np.nan

def strict_iqr(x):
    if isinstance(x, (list, np.ndarray)) and not any(pd.isna(x)):
        return np.percentile(x, 75) - np.percentile(x, 25)
    return np.nan

# Apply to SBP
df['median_30s_sbp'] = df['vector_10s_sbp'].apply(strict_median)
df['iqr_30s_sbp'] = df['vector_10s_sbp'].apply(strict_iqr)

# Apply to DBP
df['median_30s_dbp'] = df['vector_10s_dbp'].apply(strict_median)
df['iqr_30s_dbp'] = df['vector_10s_dbp'].apply(strict_iqr)

# Apply to HR
df['median_30s_hr'] = df['vector_10s_hr'].apply(strict_median)
df['iqr_30s_hr'] = df['vector_10s_hr'].apply(strict_iqr)

## Removign the ppg cases which has negetive sqi 


In [ ]:
import numpy as np
df_events2=df_merged
# 3. Check if all values in vector_10s_pleth_sqi are 0 or 1
def is_valid_sqi(triple):
    triple_array = np.array(triple)
    return np.all(np.isin(triple_array, [0, 1])) if isinstance(triple, (list, tuple)) else False
mask = (
    (df_events2['vector_10s_pleth_sqi'].apply(is_valid_sqi))
)

# 5. Apply mask
df_selected3 = df_events2[mask].copy()
df_selected3

## Startifying


In [ ]:
## focusin only oneth cardivacualr related cases in Icd10
df_events=df2

import numpy as np
bins = [0, 18, 35, 50, 65, 85, np.inf]
labels = ['0-18', '19-35', '36-50', '51-65', '66-85', '85+']
df_events["age_binned"] = pd.cut(df_events["age"], bins=bins, labels=labels, right=False)

def filter_cardiovascular_codes(icd_list):
    return [code for code in icd_list if code.startswith("I")]

df_events["cv_icd10"] = df_events["icd10_truncated"].apply(filter_cardiovascular_codes)

#df_events["label_for_stratification"] = df_events.apply(
#    lambda row: [row["event_rhythm"], str(row["age_binned"]), row["gender"]] + row["icd10_truncated"],
#    axis=1
#)

df_events["label_for_stratification"] = df_events.apply(
    lambda row: [row["event_rhythm"], str(row["age_binned"]), row["gender"]] + row["cv_icd10"],
    axis=1
)

In [ ]:
import numpy as np
from stratify import *

strat_folds = stratified_subsets(
    df_events,
    "label_for_stratification",
    [0.04] * 12,## in original it is 25 but we reduced it to 12 since be sure that all fold has all rheythm
    col_group="subject_id",
    random_seed=42
)
df_events["strat_fold"] = strat_folds
df_events.strat_fold.value_counts()
df_events.drop(["age_binned","label_for_stratification"],axis=1,inplace=True)